# Running Scenarios with PyETM

This basic workbook demonstrates:
1. Load (or create) scenarios based on an Excel input file
2. Update scenarios on the ETM platform
3. Export results to an Excel output file

For more guides and examples, see the docs at https://quintel.github.io/pyetm/user-guide/

## Setup

First, import the necessary modules and set up the paths.

In [ ]:
from pyetm import Scenarios
from pathlib import Path

## Load Scenarios from Excel

The `Scenarios.from_excel()` method reads an Excel file containing scenario definitions.
The Excel file should have:

- **MAIN sheet**: Lists scenarios with their identifiers (session IDs or saved scenario IDs)
- **INPUTS sheet**: Input parameter values (slider settings) to apply
- **SORTABLES sheet**: Technology ordering configurations (optional)
- **CUSTOM_CURVES sheet**: Custom price/demand curves (optional)
- **USERS sheet**: User permissions for saved scenarios (optional)
- **EXPORT_CONFIG sheet**: Configuration for what data to export

When `update=True` is set, the method will:
1. Load all scenarios defined in the MAIN sheet
2. Apply any changes from INPUTS, SORTABLES, CUSTOM_CURVES, and USERS sheets
3. Upload these changes to the ETM platform via API calls

The `Scenarios` object can handle both:
- **Session objects**: Temporary scenarios (identified by session IDs)
- **SavedScenario objects**: Persistent scenarios in MyETM (identified by saved scenario IDs)

In [ ]:
# Define the input file path
input_path = Path("excel/example_input_excel.xlsx")

# Load scenarios from Excel with update=True to push changes to ETM
print(f"Loading scenarios from: {input_path}")
scenarios = Scenarios.from_excel(input_path, update=True)

## Explore the Loaded Scenarios

The `Scenarios` object is a collection that you can iterate over, index into, and query.
Let's see what scenarios were loaded:

In [ ]:
# Show how many scenarios were loaded
print(f"\nLoaded {len(scenarios)} scenario(s):")

# List each scenario with its title and ID
for i, scenario in enumerate(scenarios, 1):
    print(f"  {i}. {scenario.title} (ID: {scenario.id})")

## Export Results to Excel

The `to_excel()` method exports scenario data according to the EXPORT_CONFIG sheet.
It can export:

- Input values (slider settings)
- GQuery results (calculated outputs)
- Hourly curves (electricity, heat, hydrogen, methane profiles)
- Annual exports (energy flow, sankey diagrams, production data)
- Sortables (technology ordering)
- Custom curves
- User permissions

By default, results are exported to `{input_filename}_results.xlsx` in the same directory.
Depending on your EXPORT_CONFIG settings, additional files may be created:
- `{output_filename}_hourly_curves.xlsx`: Hourly time-series data
- `{output_filename}_annual_exports.xlsx`: Annual export data

In [ ]:
# Define output path
output_path = input_path.parent / f"{input_path.stem}_results{input_path.suffix}"

# Export results to Excel
print(f"\nExporting results to: {output_path}")
scenarios.to_excel(output_path)

## Accessing Scenario Data in a notebook

You can access individual scenarios and their data programmatically:

In [ ]:
# Access the first scenario
if len(scenarios) > 0:
    first_scenario = scenarios[0]

    print(f"\nFirst scenario: {first_scenario.title}")
    print(f"  Area: {first_scenario.area_code}")
    print(f"  End year: {first_scenario.end_year}")
    print(f"  ID: {first_scenario.id}")

    # Access scenario inputs (lazy-loaded)
    inputs = first_scenario.inputs.to_dataframe(columns=['value', 'user', 'default']) # specify which columns you want

In [ ]:
# Show inputs, including how 'value' is determined user > default
inputs